# 03b. Feature Engineering Investigation and Improvement

**Project:** KPI-RAG: Explainable Root-Cause Analysis for 5G Networks
**Stage:** Phase 1 — Feature Engineering Investigation

## Why This Notebook Exists

Notebook 03 produced an unexpected result: Config A (64-dim statistics only)
outperformed Config C (582-dim full vector) on both tasks.

```
Binary F1:   A=0.981 > B=0.975 > C=0.964
Fault F1:    A=0.945 > B=0.894 > C=0.883
```

Investigation revealed the root cause: **extreme feature scale imbalance.**

TX_Bytes reaches 455,536,749 while RSRP ranges from -141 to -45.
This 10-million-fold difference causes Random Forest to ignore low-range
but diagnostically critical KPIs (RSRP, SNR, BLER) and over-index on
high-magnitude throughput channels.

## This Notebook

1. Diagnoses the scale imbalance with per-channel analysis
2. Applies selective log1p transformation to high-range channels only
3. Preserves absolute scale on diagnostically critical channels
4. Rebuilds all three ablation configs with the corrected features
5. Re-runs the ablation and compares results against Notebook 03

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

warnings.filterwarnings('ignore')

COLORS = {
    'normal':     '#2E6DB4',
    'anomaly':    '#C0392B',
    'jamming':    '#E67E22',
    'synthetic':  '#85929E',
    'axis':       '#2C3E50',
    'grid':       '#F2F4F6',
    'background': '#FFFFFF',
}

DATA_DIR = r'C:\Users\DELL\Desktop\kpi_rag\data'

KPI_NUMERICAL = [
    'RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS',
    'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes',
    'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current',
    'PRB_Utilization_DL', 'PRB_Utilization_UL',
    'UL_NumberOfPackets', 'DL_NumberOfPackets'
]

# Channels where absolute magnitude is the diagnostic signal
# These must NOT be transformed
SCALE_PRESERVE = [
    'RSRP', 'DL_BLER', 'UL_BLER', 'UL_SNR',
    'DL_MCS', 'UL_MCS', 'UL_NPRB',
    'PRBs_DL_Current', 'PRBs_UL_Current',
    'PRB_Utilization_DL', 'PRB_Utilization_UL',
]

# Channels where relative change matters but not absolute magnitude
# These are safe to log-transform
LOG_TRANSFORM = [
    'TX_Bytes', 'RX_Bytes',
    'Estimated_UL_Buffer',
    'UL_NumberOfPackets', 'DL_NumberOfPackets',
]

RF_PARAMS = {
    'n_estimators':  300,
    'max_depth':     None,
    'min_samples_leaf': 2,
    'n_jobs':        -1,
    'random_state':  42,
    'class_weight':  'balanced',
}

print('Scale-preserve channels:', SCALE_PRESERVE)
print()
print('Log-transform channels:', LOG_TRANSFORM)

## 2. Load Data

In [ ]:
X_full       = np.load(os.path.join(DATA_DIR, 'X_features.npy'))
y_binary     = np.load(os.path.join(DATA_DIR, 'y_binary.npy'))
y_multiclass = np.load(os.path.join(DATA_DIR, 'y_multiclass.npy'))
train_idx    = np.load(os.path.join(DATA_DIR, 'train_idx.npy'))
test_idx     = np.load(os.path.join(DATA_DIR, 'test_idx.npy'))

with open(os.path.join(DATA_DIR, 'type_to_int.json')) as f:
    type_to_int = json.load(f)
int_to_type = {v: k for k, v in type_to_int.items()}

y_bin_train = y_binary[train_idx]
y_bin_test  = y_binary[test_idx]
y_mc_train  = y_multiclass[train_idx]
y_mc_test   = y_multiclass[test_idx]

print(f'X_full shape: {X_full.shape}')
print(f'Train: {len(train_idx):,}   Test: {len(test_idx):,}')

## 3. Diagnose — Per-Channel Scale Analysis

Identifies which channels have extreme ranges that dominate Random Forest splits.

In [ ]:
stats_block = X_full[train_idx, 0:64]

print(f'{"Channel":<25} {"Mean":>12} {"Std":>14} {"Max":>16} {"Transform"}')
print('-' * 80)

for i, ch in enumerate(KPI_NUMERICAL):
    col_mean = stats_block[:, i*4].mean()
    col_std  = stats_block[:, i*4].std()
    col_max  = stats_block[:, i*4].max()
    transform = 'LOG' if ch in LOG_TRANSFORM else 'PRESERVE'
    flag = ' <-- HIGH RANGE' if col_max > 1000 else ''
    print(f'{ch:<25} {col_mean:>12.2f} {col_std:>14.2f} {col_max:>16.2f}   {transform}{flag}')

## 4. Apply Selective Log1p Transformation

**Why log1p and not log?**
log1p(x) = log(1 + x) handles zero values safely — no undefined values.

**Why only 5 channels?**
The 11 scale-preserve channels carry absolute magnitude as their core diagnostic signal.
RSRP = -100 dBm means weak signal. BLER = 0.45 means 45% error rate.
These must stay as-is.

The 5 log-transform channels are traffic volume metrics.
What matters diagnostically is not that TX_Bytes = 173,242 exactly,
but whether throughput dropped by 90% — a relative change log captures perfectly.

In [ ]:
def build_corrected_feature_matrix(X_raw, kpi_channels, log_channels, scale_end=320):
    X_corrected = X_raw.copy().astype(np.float64)

    for i, ch in enumerate(kpi_channels):
        if ch not in log_channels:
            continue

        # Statistics block: 4 values per channel (mean, std, min, max)
        stats_start = i * 4
        for offset in range(4):
            col = stats_start + offset
            X_corrected[:, col] = np.log1p(np.abs(X_corrected[:, col]))

        # Patchwise scale block: 2 values per patch x 8 patches per channel
        # starts at column 64 + channel_index * 16
        if scale_end > 64:
            patch_start = 64 + i * 16
            for offset in range(16):
                col = patch_start + offset
                if col < scale_end:
                    X_corrected[:, col] = np.log1p(np.abs(X_corrected[:, col]))

        # First-order differences block: same layout starting at column 320
        if X_raw.shape[1] > 320:
            diff_start = 320 + i * 16
            for offset in range(16):
                col = diff_start + offset
                if col < 576:
                    X_corrected[:, col] = np.log1p(np.abs(X_corrected[:, col]))

    return X_corrected


print('Applying log1p transformation to high-range channels...')
X_corrected = build_corrected_feature_matrix(
    X_full, KPI_NUMERICAL, LOG_TRANSFORM, scale_end=320
)
print(f'Corrected matrix shape: {X_corrected.shape}')

## 5. Verify Transformation — Per-Channel Check

In [ ]:
stats_corrected = X_corrected[train_idx, 0:64]

print(f'{"Channel":<25} {"Before Max":>14} {"After Max":>14} {"Transform"}')
print('-' * 70)

stats_original = X_full[train_idx, 0:64]

for i, ch in enumerate(KPI_NUMERICAL):
    before_max = stats_original[:, i*4].max()
    after_max  = stats_corrected[:, i*4].max()
    transform  = 'LOG' if ch in LOG_TRANSFORM else 'PRESERVE'
    print(f'{ch:<25} {before_max:>14.2f} {after_max:>14.2f}   {transform}')

## 6. Build Corrected Ablation Configs

Same three configs as Notebook 03 — but using the corrected feature matrix.

In [ ]:
X_A_v2 = X_corrected[:, :64]
X_B_v2 = X_corrected[:, :320]
X_C_v2 = X_corrected[:, :582]

X_A_train_v2, X_A_test_v2 = X_A_v2[train_idx], X_A_v2[test_idx]
X_B_train_v2, X_B_test_v2 = X_B_v2[train_idx], X_B_v2[test_idx]
X_C_train_v2, X_C_test_v2 = X_C_v2[train_idx], X_C_v2[test_idx]

print('Corrected feature matrix slices:')
print(f'  Config A v2: {X_A_v2.shape}')
print(f'  Config B v2: {X_B_v2.shape}')
print(f'  Config C v2: {X_C_v2.shape}')

## 7. Run Corrected Ablation — Binary Anomaly Detector

In [ ]:
def evaluate_binary(X_train, X_test, y_train, y_test, label):
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    f1  = f1_score(y_test, y_pred, pos_label=1)
    pre = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    print(f'  {label}: F1={f1:.4f}  Precision={pre:.4f}  Recall={rec:.4f}')
    return {'label': label, 'f1': f1, 'precision': pre, 'recall': rec, 'model': rf}


print('Corrected Binary Ablation:')
binary_v2 = []
for label, X_tr, X_te in [
    ('A-v2 (64-dim)',  X_A_train_v2, X_A_test_v2),
    ('B-v2 (320-dim)', X_B_train_v2, X_B_test_v2),
    ('C-v2 (582-dim)', X_C_train_v2, X_C_test_v2),
]:
    binary_v2.append(evaluate_binary(X_tr, X_te, y_bin_train, y_bin_test, label))

## 8. Run Corrected Ablation — Fault Classifier

In [ ]:
def evaluate_multiclass(X_train, X_test, y_train, y_test, label, type_to_int):
    JAMMING = type_to_int.get('Jamming', -1)
    mask_tr = (y_train > 0) & (y_train != JAMMING)
    mask_te = (y_test  > 0) & (y_test  != JAMMING)
    X_tr, y_tr = X_train[mask_tr], y_train[mask_tr]
    X_te, y_te = X_test[mask_te],  y_test[mask_te]
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr, y_tr)
    y_pred   = rf.predict(X_te)
    f1_macro = f1_score(y_te, y_pred, average='macro', zero_division=0)
    accuracy = (y_pred == y_te).mean()
    print(f'  {label}: F1-macro={f1_macro:.4f}  Accuracy={accuracy:.4f}')
    return {'label': label, 'f1_macro': f1_macro, 'accuracy': accuracy, 'model': rf}


print('Corrected Fault Classifier Ablation:')
mc_v2 = []
for label, X_tr, X_te in [
    ('A-v2 (64-dim)',  X_A_train_v2, X_A_test_v2),
    ('B-v2 (320-dim)', X_B_train_v2, X_B_test_v2),
    ('C-v2 (582-dim)', X_C_train_v2, X_C_test_v2),
]:
    mc_v2.append(evaluate_multiclass(X_tr, X_te, y_mc_train, y_mc_test, label, type_to_int))

## 9. Compare v1 vs v2 Results

In [ ]:
# Original results from Notebook 03
binary_v1 = [
    {'label': 'A-v1', 'f1': 0.9814},
    {'label': 'B-v1', 'f1': 0.9751},
    {'label': 'C-v1', 'f1': 0.9644},
]
mc_v1 = [
    {'label': 'A-v1', 'f1_macro': 0.9449},
    {'label': 'B-v1', 'f1_macro': 0.8939},
    {'label': 'C-v1', 'f1_macro': 0.8828},
]

print('Comparison — Binary Anomaly Detector:')
print(f'{"Config":<12} {"v1 F1":>8} {"v2 F1":>8} {"Change":>8}')
print('-' * 42)
for v1, v2 in zip(binary_v1, binary_v2):
    change = v2['f1'] - v1['f1']
    arrow  = '+' if change >= 0 else ''
    print(f'{v2["label"]:<12} {v1["f1"]:>8.4f} {v2["f1"]:>8.4f} {arrow}{change:>7.4f}')

print()
print('Comparison — Fault Classifier:')
print(f'{"Config":<12} {"v1 F1":>8} {"v2 F1":>8} {"Change":>8}')
print('-' * 42)
for v1, v2 in zip(mc_v1, mc_v2):
    change = v2['f1_macro'] - v1['f1_macro']
    arrow  = '+' if change >= 0 else ''
    print(f'{v2["label"]:<12} {v1["f1_macro"]:>8.4f} {v2["f1_macro"]:>8.4f} {arrow}{change:>7.4f}')

## 10. Comparison Chart — v1 vs v2

In [ ]:
configs = ['A (64-dim)', 'B (320-dim)', 'C (582-dim)']
b_v1 = [r['f1']       for r in binary_v1]
b_v2 = [r['f1']       for r in binary_v2]
m_v1 = [r['f1_macro'] for r in mc_v1]
m_v2 = [r['f1_macro'] for r in mc_v2]

x     = np.arange(len(configs))
width = 0.3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, v1_vals, v2_vals, title, color_v1, color_v2, ylabel in [
    (axes[0], b_v1, b_v2, 'Binary Anomaly Detector',
     COLORS['normal'], COLORS['jamming'], 'F1 Score'),
    (axes[1], m_v1, m_v2, 'Fault Classifier',
     COLORS['anomaly'], COLORS['synthetic'], 'F1-macro Score'),
]:
    bars1 = ax.bar(x - width/2, v1_vals, width, label='v1 (original)',
                   color=color_v1, edgecolor='white', linewidth=0.8, alpha=0.7)
    bars2 = ax.bar(x + width/2, v2_vals, width, label='v2 (log-corrected)',
                   color=color_v2, edgecolor='white', linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(configs, fontsize=10, color=COLORS['axis'])
    ax.set_ylabel(ylabel, fontsize=11, color=COLORS['axis'])
    ax.set_ylim(0.7, 1.05)
    ax.set_title(title, fontsize=12, color=COLORS['axis'], pad=10)
    ax.set_facecolor(COLORS['grid'])
    ax.grid(axis='y', color='white', linewidth=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=9)

    for bar, val in zip(bars2, v2_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=9, color=COLORS['axis'], fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ablation_v1_vs_v2.png'), dpi=150)
plt.show()

## 11. Save Corrected Feature Matrices

In [ ]:
np.save(os.path.join(DATA_DIR, 'X_config_A_v2.npy'), X_A_v2)
np.save(os.path.join(DATA_DIR, 'X_config_B_v2.npy'), X_B_v2)
np.save(os.path.join(DATA_DIR, 'X_config_C_v2.npy'), X_C_v2)
np.save(os.path.join(DATA_DIR, 'X_features_v2.npy'), X_corrected)

print('Saved corrected feature matrices:')
print(f'  X_features_v2.npy    {X_corrected.shape}  full corrected matrix')
print(f'  X_config_A_v2.npy   {X_A_v2.shape}')
print(f'  X_config_B_v2.npy   {X_B_v2.shape}')
print(f'  X_config_C_v2.npy   {X_C_v2.shape}')

## 12. Validation Summary

In [ ]:
checks = {
    'Corrected matrix saved':             os.path.exists(os.path.join(DATA_DIR, 'X_features_v2.npy')),
    'Config A v2 saved':                  os.path.exists(os.path.join(DATA_DIR, 'X_config_A_v2.npy')),
    'Config B v2 saved':                  os.path.exists(os.path.join(DATA_DIR, 'X_config_B_v2.npy')),
    'Config C v2 saved':                  os.path.exists(os.path.join(DATA_DIR, 'X_config_C_v2.npy')),
    'No NaN in corrected matrix':         not np.isnan(X_corrected).any(),
    'No Inf in corrected matrix':         not np.isinf(X_corrected).any(),
    'Binary v2 has 3 results':            len(binary_v2) == 3,
    'Fault v2 has 3 results':             len(mc_v2) == 3,
    'TX_Bytes max < 20 after transform':  X_corrected[train_idx, 7*4].max() < 20,
    'RSRP unchanged after transform':     abs(X_corrected[train_idx, 0].mean() -
                                              X_full[train_idx, 0].mean()) < 0.01,
}

all_passed = True
for check, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {check}')
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed.')
    print('Compare v1 vs v2 results to decide final feature matrix.')
else:
    print('One or more checks failed. Investigate before proceeding.')